# Lesson 8: Choose units

**You will learn:** how to choose units by function (a localizer), how to build the matched random control, and how to define a region from units in several layers.
**Requires:** CPU only, no downloads, under a minute. Builds on [Lessons 5](05_layers_and_regions.ipynb) and [7](07_perturb_a_model.ipynb).

In neuroscience, a *functional localizer* finds the voxels or neurons that respond more to one category than another, for example faces versus objects. The selected population is then studied or silenced. Two questions decide whether such a result is meaningful:

1. **Were the units chosen for a reason?** A localizer contrasts two stimulus categories.
2. **Is the effect specific?** Silencing the same number of *randomly chosen* units from the same population is the control. If random units have the same effect, the localizer did not find anything special.

Brain-Score expresses unit choices as a `UnitSelection` that is resolved against a model into a concrete `Selection`:

| Recipe | Chooses |
| --- | --- |
| `IndexSelection(layer, indices)` | Exactly the units you list. |
| `RandomSelection(layer, n_units, n_total, seed)` | Units drawn at random; the control. |
| `FunctionalSelection(recording_target, localizer_stimuli, contrast, n_units)` | The units that respond most to one category over another. |

In [1]:
import warnings
warnings.filterwarnings('ignore')  # keep the lesson output short

In [2]:
import tempfile, pathlib
import numpy as np
from PIL import Image
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

# Eight small synthetic images: four bright, four dark. They stand in for real stimuli.
folder = pathlib.Path(tempfile.mkdtemp())
rng = np.random.default_rng(0)
rows = []
for i in range(8):
    bright = i % 2 == 0
    low, high = (150, 255) if bright else (0, 100)
    path = folder / f'image_{i}.png'
    Image.fromarray(rng.integers(low, high, (16, 16, 3), dtype=np.uint8)).save(path)
    rows.append({'stimulus_id': f'image_{i}', 'image_path': str(path),
                 'label': 'bright' if bright else 'dark'})

stimuli = StimulusSet(rows)
stimuli.identifier = 'toy-bright-dark'
stimuli.stimulus_paths = dict(zip(stimuli['stimulus_id'], stimuli['image_path']))
stimuli

,stimulus_id,image_path,label
0,image_0,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
1,image_1,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark
2,image_2,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
3,image_3,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark
4,image_4,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
5,image_5,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark
6,image_6,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
7,image_7,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark


In [3]:
import torch

torch.manual_seed(0)
# A tiny, untrained convolutional network. Its weights are random.
net = torch.nn.Sequential(
    torch.nn.Conv2d(3, 4, kernel_size=3),  # layer '0'
    torch.nn.ReLU(),                       # layer '1'
    torch.nn.AdaptiveAvgPool2d(2),         # layer '2'
    torch.nn.Flatten(),                    # layer '3'
    torch.nn.Linear(16, 8),                # layer '4'
).eval()

def preprocess(paths):
    """Load image files into a (batch, 3, height, width) float array."""
    images = [np.asarray(Image.open(p).convert('RGB'), dtype=np.float32) / 255 for p in paths]
    return np.stack(images).transpose(0, 3, 1, 2)

In [4]:
from brainscore_core import BrainScoreModel
from brainscore.model_helpers.vision_wrapper import VisionWrapper
from brainscore.perturbation import build_pytorch_ablation_fn

model = BrainScoreModel(
    identifier='toy-cnn',
    model=net,
    extractor={'vision': VisionWrapper(identifier='toy-cnn', model=net, preprocessing=preprocess)},
    region_layer_map={'IT': '4'},
    intervention=build_pytorch_ablation_fn(net),
)

## Localize "bright-preferring" units

In [5]:
from brainscore_core import FunctionalSelection

localizer = FunctionalSelection(recording_target='IT', localizer_stimuli=stimuli,
                                contrast=(['bright'], ['dark']), n_units=2)
selected = localizer.resolve(model)
selected.layer, selected.indices

activations:   0%|          | 0/64 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/1 [00:00<?, ?it/s]

('4', [0, 7])

## Build the matched random control

Draw the same number of units from the same population that the localizer chose from. The population is stored on the resolved selection.

In [6]:
from brainscore_core import RandomSelection

population = selected.metadata['unit_population']
control = RandomSelection(layer=selected.layer, n_units=len(selected.indices),
                          n_total=len(population), population=population, seed=1).resolve(model)
control.indices

[2, 7]

## Compare the two lesions

The measure is how strongly IT distinguishes bright from dark images: for each unit, the absolute difference between its mean response to bright and to dark images, averaged over units. Record, silence, record again, and restore.

In [7]:
from brainscore_core import Perturbation, StateChange
from brainscore_core.streaming import BufferedSession

def bright_dark_selectivity():
    session = BufferedSession(stimulus_set=stimuli, requested_output_channels=['neural:IT'])
    model.participate(session)
    values = session.collect().values
    is_bright = (stimuli['label'] == 'bright').values
    return np.abs(values[is_bright].mean(axis=0) - values[~is_bright].mean(axis=0)).mean()

baseline = bright_dark_selectivity()
results = {}
for name, selection in [('localized', selected), ('random', control)]:
    model.process(StateChange(kind='ablation', target=selection, perturbation=Perturbation(kind='zero')))
    results[name] = bright_dark_selectivity()
    model.reset()

print(f'no lesion:        {baseline:.4f}')
for name, value in results.items():
    print(f'{name + " lesion:":<17} {value:.4f}  ({100 * (value - baseline) / baseline:+.0f}%)')

activations:   0%|          | 0/64 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/1 [00:00<?, ?it/s]

activations:   0%|          | 0/64 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/1 [00:00<?, ?it/s]

activations:   0%|          | 0/64 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/1 [00:00<?, ?it/s]

no lesion:        0.0200
localized lesion: 0.0132  (-34%)
random lesion:    0.0154  (-23%)


With a random, untrained network and eight images, these numbers only demonstrate the procedure. In a real study, use many stimuli, held-out localizer images, and several random seeds for the control.

## Regions built from several layers

Sometimes a brain region is better modeled by units drawn from more than one layer. `CompositeSelector` lists `(layer, indices)` pairs; `None` takes every unit of that layer. It is used as a value in `region_layer_map`.

In [8]:
from brainscore_core import CompositeSelector

composite_model = BrainScoreModel(
    identifier='toy-cnn-composite',
    model=net,
    extractor={'vision': VisionWrapper(identifier='toy-cnn-composite', model=net, preprocessing=preprocess)},
    region_layer_map={'mixed': CompositeSelector(layers=(('3', (0, 1, 2)), ('4', None)))},
)
session = BufferedSession(stimulus_set=stimuli, requested_output_channels=['neural:mixed'])
composite_model.participate(session)
mixed = session.collect()
mixed.shape, sorted(set(mixed['layer'].values))

activations:   0%|          | 0/64 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/2 [00:00<?, ?it/s]

((8, 11), ['3', '4'])

## Summary

- A `UnitSelection` is a recipe; `resolve(model)` turns it into a concrete `Selection` that recording and perturbation both use.
- Always pair a localized lesion with a matched `RandomSelection` from the same population.
- `CompositeSelector` defines a region from units in several layers.

**Next:** [Lesson 9](09_closed_loop.ipynb) lets the subject's outputs change its next input.